In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Main table from your screenshot
TABLE_NAME = "workspace.default.gold_recommender_dataset"

df = spark.table(TABLE_NAME)

display(df.limit(10))
print("Rows:", df.count())
print("Columns:", len(df.columns))
print(df.columns)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# SETTINGS
TEST_FRAC = 0.2
SEED = 42
#MIN_USER_REVIEWS = 30
#MIN_ITEM_REVIEWS = 120
MIN_USER_REVIEWS = 5
MIN_ITEM_REVIEWS = 20
MIN_CO_RATINGS = 5
#MIN_CO_RATINGS = 3
TOP_K_NEIGHBORS = 10
#TOP_K_NEIGHBORS = 30
MIN_SIMILARITY = 0.05


HOLDOUT_TABLE = "workspace.default.sephora_test_holdout"

print("Configured per-user split:", int((1-TEST_FRAC)*100), "/", int(TEST_FRAC*100))
print("Min user reviews:", MIN_USER_REVIEWS)
print("Min item reviews:", MIN_ITEM_REVIEWS)
print("Min co-ratings:", MIN_CO_RATINGS)
print("Top-K neighbors:", TOP_K_NEIGHBORS)


In [0]:
print("Holdout table will be written after the train/test split is created:", HOLDOUT_TABLE)


In [0]:
from pyspark.sql import functions as F

# Dynamically pick the available content columns so the notebook is more robust in Databricks
available_cols = set(df.columns)

def first_existing(candidates, alias):
    for c in candidates:
        if c in available_cols:
            return F.col(c).cast("string").alias(alias)
    return F.lit(None).cast("string").alias(alias)

ratings_raw = (
    df.select(
        F.col("user_id").cast("string").alias("user_id"),
        F.col("product_id").cast("string").alias("item_id"),
        F.col("rating").cast("double").alias("rating"),
        F.col("product_name").cast("string").alias("product_name"),
        F.col("brand_name").cast("string").alias("brand_name"),
        F.col("is_recommended").cast("string").alias("is_recommended"),
        F.col("ingredients").cast("string").alias("ingredients"),
        F.col("review_title").cast("string").alias("review_title"),
        F.col("review_text").cast("string").alias("review_text")
    )
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("item_id").isNotNull())
    .filter(F.col("rating").isNotNull())
)

display(ratings_raw.limit(10))
print("Ratings rows:", ratings_raw.count())


In [0]:
# Remove impossible ratings and duplicate user-item pairs
# If a user reviewed the same product multiple times, keep the mean rating
ratings_clean = (
    ratings_raw
    .filter((F.col("rating") >= 1.0) & (F.col("rating") <= 5.0))
    .groupBy("user_id", "item_id")
    .agg(
        F.avg("rating").alias("rating"),
        F.first("product_name", ignorenulls=True).alias("product_name"),
        F.first("ingredients", ignorenulls=True).alias("ingredients"),
        F.first("brand_name", ignorenulls=True).alias("brand_name"),
        F.first("is_recommended", ignorenulls=True).alias("is_recommended"),
        F.first("review_title", ignorenulls=True).alias("review_title"),
        F.first("review_text", ignorenulls=True).alias("review_text")
    )
)

display(ratings_clean.limit(10))
print("Clean ratings rows:", ratings_clean.count())

In [0]:
# Minimum activity thresholds are applied BEFORE splitting
active_users = (
    ratings_clean.groupBy("user_id")
    .count()
    .filter(F.col("count") >= MIN_USER_REVIEWS)
    .select("user_id")
)

popular_items = (
    ratings_clean.groupBy("item_id")
    .count()
    .filter(F.col("count") >= MIN_ITEM_REVIEWS)
    .select("item_id")
)

ratings_filtered = (
    ratings_clean.join(active_users, on="user_id", how="inner")
    .join(popular_items, on="item_id", how="inner")
)

# Per-user 80/20 random split so every user keeps history in train
w_split = Window.partitionBy("user_id").orderBy(F.rand(SEED))

ratings_split = (
    ratings_filtered
    .withColumn("row_num", F.row_number().over(w_split))
    .withColumn("user_count", F.count("*").over(Window.partitionBy("user_id")))
    .withColumn(
        "test_cutoff",
        F.when(
            F.col("user_count") >= 2,
            F.greatest(F.lit(1), F.floor(F.col("user_count") * F.lit(TEST_FRAC)).cast("int"))
        ).otherwise(F.lit(0))
    )
    .withColumn("is_test", F.col("row_num") <= F.col("test_cutoff"))
)

ratings_train = (
    ratings_split
    .filter(~F.col("is_test"))
    .drop("row_num", "user_count", "test_cutoff", "is_test")
)

ratings_test = (
    ratings_split
    .filter(F.col("is_test"))
    .drop("row_num", "user_count", "test_cutoff", "is_test")
)

# Continue the notebook using TRAIN only
ratings = ratings_train
spark.sql(f"DROP TABLE IF EXISTS {HOLDOUT_TABLE}")
ratings_test.write.format("delta").mode("overwrite").saveAsTable(HOLDOUT_TABLE)

print("Filtered ratings rows (full eligible set):", ratings_filtered.count())
print("Train rows:", ratings_train.count())
print("Test rows:", ratings_test.count())
print("Train users:", ratings_train.select("user_id").distinct().count())
print("Test users:", ratings_test.select("user_id").distinct().count())
print("Train items:", ratings_train.select("item_id").distinct().count())
print("Test items:", ratings_test.select("item_id").distinct().count())

display(ratings.limit(10))
display(ratings_test.limit(10))


In [0]:
display(ratings_raw.limit(10))

In [0]:
# Mean-center TRAIN ratings by user to reduce user scoring bias
user_avg = ratings.groupBy("user_id").agg(F.avg("rating").alias("user_avg_rating"))

ratings_centered = (
    ratings.join(user_avg, on="user_id", how="left")
    .withColumn("rating_centered", F.col("rating") - F.col("user_avg_rating"))
)

display(ratings_centered.limit(10))


In [0]:
# Self-join on user_id to find item pairs rated by the same user in TRAIN only
# Keep only item_a < item_b to avoid duplicates
item_pairs = (
    ratings_centered.alias("a")
    .join(ratings_centered.alias("b"), on="user_id")
    .filter(F.col("a.item_id") < F.col("b.item_id"))
    .select(
        F.col("user_id"),
        F.col("a.item_id").alias("item_a"),
        F.col("b.item_id").alias("item_b"),
        F.col("a.product_name").alias("product_a_name"),
        F.col("b.product_name").alias("product_b_name"),
        F.col("a.rating_centered").alias("rating_a"),
        F.col("b.rating_centered").alias("rating_b")
    )
)

display(item_pairs.limit(10))
print("Item pair rows:", item_pairs.count())


In [0]:
# Cosine similarity on TRAIN centered ratings
item_similarity = (
    item_pairs
    .groupBy("item_a", "item_b", "product_a_name", "product_b_name")
    .agg(
        F.sum(F.col("rating_a") * F.col("rating_b")).alias("dot_product"),
        F.sqrt(F.sum(F.col("rating_a") * F.col("rating_a"))).alias("norm_a"),
        F.sqrt(F.sum(F.col("rating_b") * F.col("rating_b"))).alias("norm_b"),
        F.count("*").alias("co_rating_count")
    )
    .withColumn(
        "similarity",
        F.when(
            (F.col("norm_a") > 0) & (F.col("norm_b") > 0),
            F.col("dot_product") / (F.col("norm_a") * F.col("norm_b"))
        ).otherwise(F.lit(0.0))
    )
)

# Keep only reliable similarities
item_similarity_filtered = (
    item_similarity
    .filter(F.col("co_rating_count") >= MIN_CO_RATINGS)
    .filter(F.col("similarity") >= MIN_SIMILARITY)
    .filter(F.col("similarity").isNotNull())
)

display(item_similarity_filtered.orderBy(F.desc("similarity")).limit(20))
print("Similarity rows:", item_similarity_filtered.count())


In [0]:
sim_ab = item_similarity_filtered.select(
    F.col("item_a").alias("item_id"),
    F.col("item_b").alias("similar_item_id"),
    F.col("product_a_name").alias("item_name"),
    F.col("product_b_name").alias("similar_item_name"),
    F.col("similarity"),
    F.col("co_rating_count")
)

sim_ba = item_similarity_filtered.select(
    F.col("item_b").alias("item_id"),
    F.col("item_a").alias("similar_item_id"),
    F.col("product_b_name").alias("item_name"),
    F.col("product_a_name").alias("similar_item_name"),
    F.col("similarity"),
    F.col("co_rating_count")
)

item_neighbors = sim_ab.unionByName(sim_ba)

display(item_neighbors.orderBy(F.desc("similarity")).limit(20))


In [0]:
w = Window.partitionBy("item_id").orderBy(F.desc("similarity"), F.desc("co_rating_count"))

item_neighbors_topk = (
    item_neighbors
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= TOP_K_NEIGHBORS)
    .drop("rank")
)

display(item_neighbors_topk.limit(20))
print("Top-K neighbor rows:", item_neighbors_topk.count())


In [0]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, Normalizer
from pyspark.ml import Pipeline

# Build lighter item-level content features from TRAIN only
# To keep this workable in Databricks Free Edition, cap concatenated review text length.
item_text_train = (
    ratings
    .groupBy("item_id")
    .agg(
        F.first("product_name", ignorenulls=True).alias("product_name"),
        F.first("ingredients", ignorenulls=True).alias("ingredients"),
        F.first("brand_name", ignorenulls=True).alias("brand_name"),
        F.substring(F.concat_ws(" ", F.collect_list(F.coalesce(F.col("review_title"), F.lit("")))), 1, 2000).alias("review_titles_blob"),
        F.substring(F.concat_ws(" ", F.collect_list(F.coalesce(F.col("review_text"), F.lit("")))), 1, 4000).alias("review_text_blob")
    )
    .withColumn(
        "combined_text",
        F.concat_ws(
            " ",
            F.coalesce(F.col("product_name"), F.lit("")),
            F.coalesce(F.col("brand_name"), F.lit("")),
            F.coalesce(F.col("ingredients"), F.lit("")),
            F.coalesce(F.col("review_titles_blob"), F.lit("")),
            F.coalesce(F.col("review_text_blob"), F.lit(""))
        )
    )
)

text_pipeline = Pipeline(stages=[
    RegexTokenizer(inputCol="combined_text", outputCol="tokens", pattern="\\W+"),
    StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens"),
    HashingTF(inputCol="filtered_tokens", outputCol="tf_features", numFeatures=1024),
    IDF(inputCol="tf_features", outputCol="tfidf_features"),
    Normalizer(inputCol="tfidf_features", outputCol="norm_features", p=2.0)
])

text_model = text_pipeline.fit(item_text_train)
item_text_features = text_model.transform(item_text_train).select("item_id", "combined_text", "norm_features")

display(item_text_features.limit(10))
print("Item text feature rows:", item_text_features.count())


In [0]:
ratings.printSchema()

spark.table("workspace.default.gold_recommender_dataset").printSchema()

In [0]:
from pyspark.ml.functions import vector_to_array

# Compute content similarity only for the TRAIN top-K collaborative neighbors
# This avoids expensive all-to-all content similarity in the free Databricks environment.
item_features_arr = item_text_features.select(
    "item_id",
    vector_to_array("norm_features").alias("feature_arr")
)

hybrid_neighbors_topk = (
    item_neighbors_topk.alias("n")
    .join(
        item_features_arr.alias("i"),
        F.col("n.item_id") == F.col("i.item_id"),
        how="left"
    )
    .join(
        item_features_arr.alias("j"),
        F.col("n.similar_item_id") == F.col("j.item_id"),
        how="left"
    )
    .withColumn(
        "content_similarity",
        F.when(
            F.col("i.feature_arr").isNotNull() & F.col("j.feature_arr").isNotNull(),
            F.expr("""
                aggregate(
                    zip_with(i.feature_arr, j.feature_arr, (x, y) -> x * y),
                    0D,
                    (acc, x) -> acc + x
                )
            """)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "hybrid_similarity",
        0.7 * F.col("similarity") + 0.3 * F.col("content_similarity")
    )
    .select(
        "n.item_id",
        "n.similar_item_id",
        "n.item_name",
        "n.similar_item_name",
        "n.co_rating_count",
        "similarity",
        "content_similarity",
        "hybrid_similarity"
    )
)

display(hybrid_neighbors_topk.orderBy(F.desc("hybrid_similarity")).limit(20))

hybrid_neighbors_topk.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.sephora_hybrid_item_neighbors_70_30"
)

print("Saved table: workspace.default.sephora_hybrid_item_neighbors_70_30")


In [0]:
TARGET_ITEM_ID = "P420652"   # replace with any product_id you want

similar_items = (
    spark.table("workspace.default.sephora_hybrid_item_neighbors_70_30")
    .filter(F.col("item_id") == TARGET_ITEM_ID)
    .orderBy(F.desc("hybrid_similarity"), F.desc("co_rating_count"))
)

display(similar_items.limit(20))


In [0]:
# Replace with a real user_id from your TRAIN data
TARGET_USER_ID = "10004195528"
# 1741593524
# 10725439731
ratings_used = ratings.select("user_id", "item_id", "rating")
neighbors = spark.table("workspace.default.sephora_hybrid_item_neighbors_70_30")

# Items already rated by the user in TRAIN
user_history = ratings_used.filter(F.col("user_id") == TARGET_USER_ID)

display(user_history)

# Join user's rated items to their similar neighbors
candidate_scores = (
    user_history.alias("u")
    .join(
        neighbors.alias("n"),
        F.col("u.item_id") == F.col("n.item_id"),
        how="inner"
    )
    .select(
        F.col("u.user_id"),
        F.col("n.similar_item_id").alias("candidate_item_id"),
        F.col("n.similar_item_name").alias("candidate_item_name"),
        F.col("u.rating"),
        F.col("n.hybrid_similarity").alias("hybrid_similarity"),
        (F.col("u.rating") * F.col("n.hybrid_similarity")).alias("weighted_score")
    )
)

# Remove items the user already rated
already_rated = user_history.select(F.col("item_id").alias("candidate_item_id"))

user_recommendations = (
    candidate_scores
    .join(already_rated, on="candidate_item_id", how="left_anti")
    .groupBy("user_id", "candidate_item_id", "candidate_item_name")
    .agg(
        F.sum("weighted_score").alias("score_num"),
        F.sum(F.abs("hybrid_similarity")).alias("score_den")
    )
    .withColumn(
        "predicted_rating",
        F.when(F.col("score_den") > 0, F.col("score_num") / F.col("score_den"))
         .otherwise(F.lit(None))
    )
    .orderBy(F.desc("predicted_rating"))
)

display(user_recommendations.limit(20))


In [0]:
def get_similar_items(item_id, top_n=10):
    sim_df = spark.table("workspace.default.sephora_hybrid_item_neighbors_70_30")
    result = (
        sim_df.filter(F.col("item_id") == str(item_id))
              .orderBy(F.desc("hybrid_similarity"), F.desc("co_rating_count"))
              .limit(top_n)
    )
    return result

def recommend_for_user(user_id, top_n=10):
    ratings_used = ratings.select("user_id", "item_id", "rating")
    neighbors = spark.table("workspace.default.sephora_hybrid_item_neighbors_70_30")

    user_history = ratings_used.filter(F.col("user_id") == str(user_id))
    already_rated = user_history.select(F.col("item_id").alias("candidate_item_id"))

    candidate_scores = (
        user_history.alias("u")
        .join(neighbors.alias("n"), F.col("u.item_id") == F.col("n.item_id"), "inner")
        .select(
            F.col("u.user_id"),
            F.col("n.similar_item_id").alias("candidate_item_id"),
            F.col("n.similar_item_name").alias("candidate_item_name"),
            F.col("u.rating"),
            F.col("n.hybrid_similarity").alias("hybrid_similarity"),
            (F.col("u.rating") * F.col("n.hybrid_similarity")).alias("weighted_score")
        )
    )

    recs = (
        candidate_scores
        .join(already_rated, on="candidate_item_id", how="left_anti")
        .groupBy("user_id", "candidate_item_id", "candidate_item_name")
        .agg(
            F.sum("weighted_score").alias("score_num"),
            F.sum(F.abs("hybrid_similarity")).alias("score_den")
        )
        .withColumn(
            "predicted_rating",
            F.when(F.col("score_den") > 0, F.col("score_num") / F.col("score_den"))
             .otherwise(F.lit(None))
        )
        .orderBy(F.desc("predicted_rating"))
        .limit(top_n)
    )
    return recs


In [0]:
# Example 1: similar items for a product
display(get_similar_items("P420652", top_n=10))

# Example 2: recommendations for a user
display(recommend_for_user("10725439731", top_n=10))


In [0]:
user_recommendations.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.user_recommendations_hybrid_70_30"
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Holdout scoring for HYBRID using TRAIN-built collaborative neighbors plus content similarity
train_history = ratings_centered.select("user_id", "item_id", "rating_centered")
user_avg_test = user_avg.select("user_id", "user_avg_rating")
neighbors = hybrid_neighbors_topk.select(
    "item_id",
    F.col("similar_item_id").alias("neighbor_item_id"),
    F.col("hybrid_similarity").alias("similarity")
)

# For each test pair (user, target item), look up hybrid-scored neighbor items the user rated in TRAIN
candidate_contribs = (
    ratings_test.alias("t")
    .join(neighbors.alias("n"), F.col("t.item_id") == F.col("n.item_id"), how="left")
    .join(
        train_history.alias("h"),
        (F.col("t.user_id") == F.col("h.user_id")) &
        (F.col("n.neighbor_item_id") == F.col("h.item_id")),
        how="left"
    )
    .filter(F.col("h.rating_centered").isNotNull())
    .select(
        F.col("t.user_id"),
        F.col("t.item_id"),
        F.col("t.product_name"),
        F.col("t.rating").alias("actual_rating"),
        F.col("n.similarity"),
        F.col("h.rating_centered"),
        (F.col("n.similarity") * F.col("h.rating_centered")).alias("weighted_centered")
    )
)

pred_centered = (
    candidate_contribs
    .groupBy("user_id", "item_id", "product_name", "actual_rating")
    .agg(
        F.sum("weighted_centered").alias("score_numerator"),
        F.sum(F.abs("similarity")).alias("score_denominator"),
        F.count("*").alias("support")
    )
    .withColumn(
        "pred_centered",
        F.when(F.col("score_denominator") > 0,
               F.col("score_numerator") / F.col("score_denominator"))
         .otherwise(F.lit(None))
    )
)

df_pred = (
    ratings_test.alias("t")
    .join(pred_centered.alias("p"), on=["user_id", "item_id", "product_name"], how="left")
    .join(user_avg_test, on="user_id", how="left")
    .withColumn(
        "prediction_raw",
        F.when(F.col("pred_centered").isNotNull(), F.col("user_avg_rating") + F.col("pred_centered"))
         .otherwise(F.col("user_avg_rating"))
    )
    .withColumn(
        "prediction",
        F.when(F.col("prediction_raw") < 1.0, F.lit(1.0))
         .when(F.col("prediction_raw") > 5.0, F.lit(5.0))
         .otherwise(F.col("prediction_raw"))
    )
    .select(
        "user_id",
        "item_id",
        "product_name",
        F.col("t.rating").alias("actual_rating"),
        "prediction",
        F.coalesce(F.col("p.support"), F.lit(0)).alias("support")
    )
)

display(df_pred.limit(20))
print("Scored holdout rows:", df_pred.count())

# RMSE on holdout
rmse = (
    df_pred
    .select(F.sqrt(F.avg(F.pow(F.col("prediction") - F.col("actual_rating"), 2))).alias("rmse"))
    .collect()[0][0]
)

# MAE
mae = (
    df_pred
    .select(F.mean(F.abs(F.col("prediction") - F.col("actual_rating"))).alias("mae"))
    .collect()[0]["mae"]
)

# Ranking metrics from scored holdout pairs
K = 10
rank_window = Window.partitionBy("user_id").orderBy(F.desc("prediction"), F.desc("support"), F.asc("item_id"))

ranked_pred = (
    df_pred
    .withColumn("rank", F.row_number().over(rank_window))
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
)

topk = ranked_pred.filter(F.col("rank") <= K)

user_relevant_counts = (
    df_pred
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
    .groupBy("user_id")
    .agg(F.sum("relevant").alias("n_relevant"))
)

per_user_hits = (
    topk.groupBy("user_id")
    .agg(F.sum("relevant").alias("hits"))
    .join(user_relevant_counts, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn("precision_at_k", F.col("hits") / F.lit(K))
    .withColumn(
        "recall_at_k",
        F.when(F.col("n_relevant") > 0, F.col("hits") / F.col("n_relevant")).otherwise(F.lit(0.0))
    )
)

# MAP@K
ap_base = (
    topk
    .withColumn("cum_relevant", F.sum("relevant").over(rank_window.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn(
        "precision_at_rank",
        F.when(F.col("relevant") == 1, F.col("cum_relevant") / F.col("rank")).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("precision_at_rank").alias("sum_precisions"))
    .join(user_relevant_counts, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn(
        "ap_at_k",
        F.when(F.col("n_relevant") > 0, F.col("sum_precisions") / F.least(F.col("n_relevant"), F.lit(K)))
         .otherwise(F.lit(0.0))
    )
)

precision_at_k = per_user_hits.agg(F.avg("precision_at_k")).collect()[0][0]
recall_at_k = per_user_hits.agg(F.avg("recall_at_k")).collect()[0][0]
map_at_k = ap_base.agg(F.avg("ap_at_k")).collect()[0][0]

total_catalog_items = ratings_clean.select("item_id").distinct().count()

catalogue_coverage = (
    topk.select("item_id").distinct().count() /
    total_catalog_items
    if total_catalog_items > 0 else 0.0
)

# NDCG@K (SAFE VERSION)
dcg_base = (
    topk
    .withColumn(
        "dcg_component",
        F.when(
            (F.col("relevant") == 1) & (F.col("rank") > 0),
            1 / F.log2(F.col("rank") + 1)
        ).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("dcg_component").alias("dcg"))
)

ideal_dcg = (
    user_relevant_counts
    .withColumn(
        "ideal_dcg",
        F.expr(f"""
        aggregate(
            sequence(1, greatest(least(n_relevant,{K}),1)),
            0D,
            (acc,x) -> acc + 1/log2(x+1)
        )
        """)
    )
)

ndcg_df = (
    dcg_base
    .join(ideal_dcg, on="user_id", how="left")
    .withColumn(
        "ndcg_at_k",
        F.when(
            (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
            F.col("dcg") / F.col("ideal_dcg")
        ).otherwise(F.lit(0.0))
    )
)

ndcg_at_k = ndcg_df.select(F.avg("ndcg_at_k")).collect()[0][0]

random_hit_rate = df_pred.select(F.avg(F.when(F.col("actual_rating") >= 4.0, 1.0).otherwise(0.0))).collect()[0][0]
lift_over_random = precision_at_k / random_hit_rate if random_hit_rate and random_hit_rate > 0 else None

# Inter-user diversity: 1 - average Jaccard overlap across top-K item lists
# Sample users first to keep pairwise computation workable in Databricks Free Edition.
user_lists = topk.groupBy("user_id").agg(F.collect_set("item_id").alias("rec_list"))
sampled_user_lists = user_lists.orderBy(F.rand(SEED)).limit(300)

user_pairs = (
    sampled_user_lists.alias("a")
    .crossJoin(sampled_user_lists.alias("b"))
    .filter(F.col("a.user_id") < F.col("b.user_id"))
    .select(
        F.size(F.array_intersect(F.col("a.rec_list"), F.col("b.rec_list"))).alias("intersect_size"),
        F.size(F.array_union(F.col("a.rec_list"), F.col("b.rec_list"))).alias("union_size")
    )
    .withColumn(
        "inter_user_diversity",
        F.when(F.col("union_size") > 0, 1 - (F.col("intersect_size") / F.col("union_size"))).otherwise(F.lit(0.0))
    )
)
inter_user_diversity = user_pairs.agg(F.avg("inter_user_diversity")).collect()[0][0]

# Serendipity proxy: relevant recommendations that are not among the most popular training items
item_popularity = ratings.groupBy("item_id").agg(F.count("*").alias("popularity"))
pop_threshold = item_popularity.approxQuantile("popularity", [0.8], 0.01)[0] if item_popularity.count() > 0 else 0

serendipity = (
    topk.join(item_popularity, on="item_id", how="left")
    .withColumn("unexpected", F.when(F.col("popularity") <= pop_threshold, 1).otherwise(0))
    .withColumn("serendipitous", F.when((F.col("relevant") == 1) & (F.col("unexpected") == 1), 1).otherwise(0))
    .groupBy("user_id")
    .agg((F.sum("serendipitous") / F.lit(K)).alias("serendipity"))
)
serendipity_score = serendipity.agg(F.avg("serendipity")).collect()[0][0]

metrics_df = spark.createDataFrame([
    ("RMSE", float(rmse) if rmse is not None else None),
    ("MAE", float(mae) if mae is not None else None),
    (f"Precision@{K}", float(precision_at_k)),
    (f"Recall@{K}", float(recall_at_k)),
    (f"MAP@{K}", float(map_at_k)),
    (f"NDCG@{K}", float(ndcg_at_k)),
    ("Catalogue coverage", float(catalogue_coverage)),
    ("Lift over random", float(lift_over_random) if lift_over_random is not None else None),
    ("Serendipity", float(serendipity_score) if serendipity_score is not None else None),
    ("Inter-user diversity", float(inter_user_diversity) if inter_user_diversity is not None else None),
], ["metric", "value"])

display(metrics_df)


In [0]:
# Evaluate same on Train set

# For each test pair (user, target item), look up hybrid-scored neighbor items the user rated in TRAIN
candidate_contribs_train = (
    ratings.alias("t")
    .join(neighbors.alias("n"), F.col("t.item_id") == F.col("n.item_id"), how="left")
    .join(
        train_history.alias("h"),
        (F.col("t.user_id") == F.col("h.user_id")) &
        (F.col("n.neighbor_item_id") == F.col("h.item_id")),
        how="left"
    )
    .filter(F.col("h.rating_centered").isNotNull())
    .select(
        F.col("t.user_id"),
        F.col("t.item_id"),
        F.col("t.product_name"),
        F.col("t.rating").alias("actual_rating"),
        F.col("n.similarity"),
        F.col("h.rating_centered"),
        (F.col("n.similarity") * F.col("h.rating_centered")).alias("weighted_centered")
    )
)

pred_centered_train = (
    candidate_contribs_train
    .groupBy("user_id", "item_id", "product_name", "actual_rating")
    .agg(
        F.sum("weighted_centered").alias("score_numerator"),
        F.sum(F.abs("similarity")).alias("score_denominator"),
        F.count("*").alias("support")
    )
    .withColumn(
        "pred_centered",
        F.when(F.col("score_denominator") > 0,
               F.col("score_numerator") / F.col("score_denominator"))
         .otherwise(F.lit(None))
    )
)

df_pred_train = (
    ratings.alias("t")
    .join(pred_centered_train.alias("p"), on=["user_id", "item_id", "product_name"], how="left")
    .join(user_avg_test, on="user_id", how="left")
    .withColumn(
        "prediction_raw",
        F.when(F.col("pred_centered").isNotNull(), F.col("user_avg_rating") + F.col("pred_centered"))
         .otherwise(F.col("user_avg_rating"))
    )
    .withColumn(
        "prediction",
        F.when(F.col("prediction_raw") < 1.0, F.lit(1.0))
         .when(F.col("prediction_raw") > 5.0, F.lit(5.0))
         .otherwise(F.col("prediction_raw"))
    )
    .select(
        "user_id",
        "item_id",
        "product_name",
        F.col("t.rating").alias("actual_rating"),
        "prediction",
        F.coalesce(F.col("p.support"), F.lit(0)).alias("support")
    )
)

display(df_pred_train.limit(20))
print("Scored train rows:", df_pred_train.count())

# RMSE
rmse_train = (
    df_pred_train
    .select(F.sqrt(F.avg(F.pow(F.col("prediction") - F.col("actual_rating"), 2))).alias("rmse"))
    .collect()[0][0]
)

# MAE
mae_train = (
    df_pred_train
    .select(F.mean(F.abs(F.col("prediction") - F.col("actual_rating"))).alias("mae"))
    .collect()[0]["mae"]
)

# Ranking metrics

ranked_pred_train = (
    df_pred_train
    .withColumn("rank", F.row_number().over(rank_window))
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
)

topk_train = ranked_pred_train.filter(F.col("rank") <= K)

user_relevant_counts_train = (
    df_pred_train
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
    .groupBy("user_id")
    .agg(F.sum("relevant").alias("n_relevant"))
)

per_user_hits_train = (
    topk_train.groupBy("user_id")
    .agg(F.sum("relevant").alias("hits"))
    .join(user_relevant_counts_train, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn("precision_at_k", F.col("hits") / F.lit(K))
    .withColumn(
        "recall_at_k",
        F.when(F.col("n_relevant") > 0, F.col("hits") / F.col("n_relevant")).otherwise(F.lit(0.0))
    )
)

# MAP@K
ap_base_train = (
    topk_train
    .withColumn("cum_relevant", F.sum("relevant").over(rank_window.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn(
        "precision_at_rank",
        F.when(F.col("relevant") == 1, F.col("cum_relevant") / F.col("rank")).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("precision_at_rank").alias("sum_precisions"))
    .join(user_relevant_counts, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn(
        "ap_at_k",
        F.when(F.col("n_relevant") > 0, F.col("sum_precisions") / F.least(F.col("n_relevant"), F.lit(K)))
         .otherwise(F.lit(0.0))
    )
)

precision_at_k_train = per_user_hits_train.agg(F.avg("precision_at_k")).collect()[0][0]
recall_at_k_train = per_user_hits_train.agg(F.avg("recall_at_k")).collect()[0][0]
map_at_k_train = ap_base_train.agg(F.avg("ap_at_k")).collect()[0][0]

catalogue_coverage_train = (
    topk_train.select("item_id").distinct().count() /
    total_catalog_items
    if total_catalog_items > 0 else 0.0
)

# NDCG@K (SAFE VERSION)
dcg_base_train = (
    topk_train
    .withColumn(
        "dcg_component",
        F.when(
            (F.col("relevant") == 1) & (F.col("rank") > 0),
            1 / F.log2(F.col("rank") + 1)
        ).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("dcg_component").alias("dcg"))
)

ideal_dcg_train = (
    user_relevant_counts_train
    .withColumn(
        "ideal_dcg",
        F.expr(f"""
        aggregate(
            sequence(1, greatest(least(n_relevant,{K}),1)),
            0D,
            (acc,x) -> acc + 1/log2(x+1)
        )
        """)
    )
)

ndcg_df_train = (
    dcg_base_train
    .join(ideal_dcg_train, on="user_id", how="left")
    .withColumn(
        "ndcg_at_k",
        F.when(
            (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
            F.col("dcg") / F.col("ideal_dcg")
        ).otherwise(F.lit(0.0))
    )
)

ndcg_at_k_train = ndcg_df_train.select(F.avg("ndcg_at_k")).collect()[0][0]

random_hit_rate_train = df_pred_train.select(F.avg(F.when(F.col("actual_rating") >= 4.0, 1.0).otherwise(0.0))).collect()[0][0]
lift_over_random_train = precision_at_k_train / random_hit_rate_train if random_hit_rate_train and random_hit_rate_train > 0 else None

# Inter-user diversity: 1 - average Jaccard overlap across top-K item lists
# Sample users first to keep pairwise computation workable in Databricks Free Edition.
user_lists_train = topk_train.groupBy("user_id").agg(F.collect_set("item_id").alias("rec_list"))
sampled_user_lists_train = user_lists_train.orderBy(F.rand(SEED)).limit(300)

user_pairs_train = (
    sampled_user_lists_train.alias("a")
    .crossJoin(sampled_user_lists_train.alias("b"))
    .filter(F.col("a.user_id") < F.col("b.user_id"))
    .select(
        F.size(F.array_intersect(F.col("a.rec_list"), F.col("b.rec_list"))).alias("intersect_size"),
        F.size(F.array_union(F.col("a.rec_list"), F.col("b.rec_list"))).alias("union_size")
    )
    .withColumn(
        "inter_user_diversity",
        F.when(F.col("union_size") > 0, 1 - (F.col("intersect_size") / F.col("union_size"))).otherwise(F.lit(0.0))
    )
)
inter_user_diversity_train = user_pairs_train.agg(F.avg("inter_user_diversity")).collect()[0][0]

serendipity_train = (
    topk_train.join(item_popularity, on="item_id", how="left")
    .withColumn("unexpected", F.when(F.col("popularity") <= pop_threshold, 1).otherwise(0))
    .withColumn("serendipitous", F.when((F.col("relevant") == 1) & (F.col("unexpected") == 1), 1).otherwise(0))
    .groupBy("user_id")
    .agg((F.sum("serendipitous") / F.lit(K)).alias("serendipity"))
)
serendipity_score_train = serendipity_train.agg(F.avg("serendipity")).collect()[0][0]

metrics_df_train = spark.createDataFrame([
    ("RMSE", float(rmse_train) if rmse_train is not None else None),
    ("MAE", float(mae_train) if mae_train is not None else None),
    (f"Precision@{K}", float(precision_at_k_train)),
    (f"Recall@{K}", float(recall_at_k_train)),
    (f"MAP@{K}", float(map_at_k_train)),
    (f"NDCG@{K}", float(ndcg_at_k_train)),
    ("Catalogue coverage", float(catalogue_coverage_train)),
    ("Lift over random", float(lift_over_random_train) if lift_over_random_train is not None else None),
    ("Serendipity", float(serendipity_score_train) if serendipity_score_train is not None else None),
    ("Inter-user diversity", float(inter_user_diversity_train) if inter_user_diversity_train is not None else None),
], ["metric", "value"])

display(metrics_df_train)


In [0]:
display(topk_train)

In [0]:
metric_data = [
{"model_name": "Hybrid",
 "dataset": "Test",
 "rmse": (float(rmse) if rmse is not None else None),
 "mae": (float(mae) if mae is not None else None),
 "precision": (float(precision_at_k) if precision_at_k is not None else None),
 "recall": (float(recall_at_k) if recall_at_k is not None else None),
 "map": (float(map_at_k) if map_at_k is not None else None),
 "ndcg": (float(ndcg_at_k) if ndcg_at_k is not None else None),
 "coverage": (float(catalogue_coverage) if catalogue_coverage is not None else None),
 "lift": (float(lift_over_random) if lift_over_random is not None else None),
 "serendipity": (float(serendipity_score) if serendipity_score is not None else None),
 "diversity": (float(inter_user_diversity) if inter_user_diversity is not None else None),
 "k_value": K
}
]

metric_data_train = [
{"model_name": "Hybrid",
 "dataset": "Train",
 "rmse": (float(rmse_train) if rmse_train is not None else None),
 "mae": (float(mae_train) if mae_train is not None else None),
 "precision": (float(precision_at_k_train) if precision_at_k_train is not None else None),
 "recall": (float(recall_at_k_train) if recall_at_k_train is not None else None),
 "map": (float(map_at_k_train) if map_at_k_train is not None else None),
 "ndcg": (float(ndcg_at_k_train) if ndcg_at_k_train is not None else None),
 "coverage": (float(catalogue_coverage_train) if catalogue_coverage_train is not None else None),
 "lift": (float(lift_over_random_train) if lift_over_random_train is not None else None),
 "serendipity": (float(serendipity_score_train) if serendipity_score_train is not None else None),
 "diversity": (float(inter_user_diversity_train) if inter_user_diversity_train is not None else None),
 "k_value": K
}
]

model_metrics = spark.createDataFrame(metric_data).withColumn("timestamp", F.current_timestamp())
model_metrics_train = spark.createDataFrame(metric_data_train).withColumn("timestamp", F.current_timestamp())
model_metrics.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.model_metrics")
model_metrics_train.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.model_metrics")

model_eval = spark.table("workspace.default.model_metrics")
display(model_eval)

